# Global Optimization Methods

This notebook presents simple implementations of several global optimization methods:

- Random Walk (RW)
- Random Sampling / Search (RS)
- Greedy Local Search (GLS)
- Stochastic Local Search (SLS)
- Simulated Annealing (SA)
- Particle Swarm Optimization (PSO)

We begin with a very simple two-variable function, allowing the behavior of each method to be understood in a clear and controlled setting.

Once the methods have been validated, they are applied to a more challenging multimodal function.  
In this case, the landscape contains many local minima, and the choice of the initial point plays a much more significant role.

# Utilities

This section collects all shared code:

- plotting functions
- helper functions
- shared numerical settings


In [ ]:
# Import the numerical and plotting libraries.
import numpy as np
import matplotlib.pyplot as plt
# Fix the random seed so the results are reproducible.
np.random.seed(7)


In [ ]:
# Define the search box used in the first examples.
lower_bowl = np.array([-4.0, -4.0])
upper_bowl = np.array([4.0, 4.0])

# Define a common initial point for the local methods.
x0_bowl = np.array([3.0, -2.5])

In [ ]:
# Define a helper function to keep points inside the box bounds.
def clip_to_bounds(x, lower, upper):
    return np.minimum(np.maximum(x, lower), upper)

# Define a helper function to build contour data on a rectangular grid.
def build_grid_values(f, lower, upper, n_points=200):
    x1_values = np.linspace(lower[0], upper[0], n_points)
    x2_values = np.linspace(lower[1], upper[1], n_points)
    X1, X2 = np.meshgrid(x1_values, x2_values)
    Z = np.zeros_like(X1)

    # Evaluate the function point by point.
    for i in range(X1.shape[0]):
        for j in range(X1.shape[1]):
            Z[i, j] = f(np.array([X1[i, j], X2[i, j]]))

    return X1, X2, Z

# Define a helper function to plot the best value curve and the search path side by side.
def plot_progress_and_path(f, lower, upper, path, best_values, title, best_point=None, path_color="red"):
    X1, X2, Z = build_grid_values(f, lower, upper)
    path = np.array(path)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

    # Plot the best value found so far.
    axes[0].plot(best_values, color="black", linewidth=2)
    axes[0].set_xlabel("iteration")
    axes[0].set_ylabel("best value so far")
    axes[0].set_title(title + " - best value")
    axes[0].grid(True, alpha=0.3)

    # Plot the path on contour lines.
    axes[1].contour(X1, X2, Z, levels=25)
    axes[1].plot(path[:, 0], path[:, 1], marker='o', markersize=3.5, linewidth=1.5, color=path_color)
    axes[1].scatter(path[0, 0], path[0, 1], s=120, color="limegreen", edgecolor="black", label="initial", zorder=5)
    axes[1].scatter(path[-1, 0], path[-1, 1], s=120, color="gold", edgecolor="black", label="final", zorder=5)
    if best_point is not None:
        axes[1].scatter(best_point[0], best_point[1], marker="*", s=220, color="magenta", edgecolor="black", label="best point", zorder=6)
    axes[1].set_xlabel("$x_1$")
    axes[1].set_ylabel("$x_2$")
    axes[1].set_title(title + " - path")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

# Define a helper function to print a short numerical summary.
def print_result(name, best_x, best_fx):
    print(name)
    print("best point found =", np.round(best_x, 6))
    print("best value found =", round(float(best_fx), 6))
    print("-" * 50)


# Test Example: Bowl Function

We consider the following function:

\begin{equation}
f(x_1, x_2) = x_1^2 + x_2^2
\end{equation}

This is a simple convex quadratic function.

The global minimum is given by:

\begin{equation}
x^* = (0, 0)
\end{equation}

with minimum value:

\begin{equation}
f(x^*) = 0
\end{equation}

In [ ]:

# Define the simple bowl function used in the first examples.
def bowl(x):
    x1, x2 = x
    return x1**2 + x2**2


In [ ]:
# Import required tools for 3D plotting.
from mpl_toolkits.mplot3d import Axes3D

# Create a grid of points over the domain.
x1 = np.linspace(-5, 5, 200)
x2 = np.linspace(-5, 5, 200)
X1, X2 = np.meshgrid(x1, x2)

# Evaluate the function on the grid.
Z = X1**2 + X2**2

# Create a figure with two subplots.
fig = plt.figure(figsize=(12, 5))

# ---- 2D contour plot ----
ax1 = fig.add_subplot(1, 2, 1)
ax1.contour(X1, X2, Z, levels=20)
ax1.scatter(0, 0, s=100)  # Minimum point
ax1.set_title("2D Contour Plot")
ax1.set_xlabel("$x_1$")
ax1.set_ylabel("$x_2$")

# ---- 3D surface plot ----
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(X1, X2, Z)
ax2.scatter(0, 0, 0, s=100)  # Minimum point
ax2.set_title("3D Surface Plot")
ax2.set_xlabel("$x_1$")
ax2.set_ylabel("$x_2$")
ax2.set_zlabel("$f(x)$")

# Adjust layout and show.
plt.tight_layout()
plt.show()

# Random Walk (RW)

**Main idea**

Random Walk moves from the current point to a random nearby point.

At iteration $k$:
- start from the current point $x_k$
- generate a random perturbation
- move to the perturbed point
- keep the best point seen so far

The method does not force improvement at each step.

In [ ]:
# Implement Random Walk in a direct and simple way.
def random_walk(f, lower, upper, x0, step_size=0.4, n_steps=200):
    # Start from the initial point.
    x = np.array(x0, dtype=float)

    # Evaluate the function at the initial point.
    fx = f(x)

    # Store the current best point and value.
    best_x = x.copy()
    best_fx = fx

    # Store the visited path and the best values.
    path = [x.copy()]
    best_values = [best_fx]

    # Repeat the random local move.
    for _ in range(n_steps):
        # Generate a random perturbation.
        step = step_size * np.random.randn(len(x))

        # Build the new candidate point.
        candidate =  

        # Keep the candidate inside the box (use the function clip_to_bounds
        candidate =  

        # Move to the candidate.
        x =  

        # Evaluate the function at the new point.
        fx =  

        # Update the best point if needed.
        if fx < best_fx:
            best_x =  
            best_fx =  

        # Save the history.
        path.append(x.copy())
        best_values.append(best_fx)

    # Return the results.
    return best_x, best_fx, path, best_values

## Example: Random Walk on the bowl function

In [ ]:
# Run Random Walk on the bowl function.
best_x_rw, best_fx_rw, path_rw, best_values_rw = random_walk(
    bowl, lower_bowl, upper_bowl, x0_bowl, step_size=0.45, n_steps=120
)

# Print the numerical result.
print_result("Random Walk", best_x_rw, best_fx_rw)



# Plot the best value curve and the search path side by side.
plot_progress_and_path(bowl, lower_bowl, upper_bowl, path_rw, best_values_rw, "Random Walk on the bowl function", best_x_rw, path_color="red")


# Random Sampling / Search (RS)

**Main idea**

Random Sampling ignores the current point and samples directly from the whole search box.

At every iteration:
- generate a completely new point at random
- evaluate the objective
- keep the best point found so far

This is one of the simplest global optimization ideas.

In [ ]:
# Implement Random Sampling with direct uniform draws in the full box.
def random_search(f, lower, upper, n_steps=200):
    # Draw the first random point.
    x = np.random.uniform(lower, upper)

    # Evaluate the function at the first point.
    fx = f(x)

    # Store the current best point and value.
    best_x = x.copy()
    best_fx = fx

    # Store the sampled points and the best values.
    path = [x.copy()]
    best_values = [best_fx]

    # Repeat the random sampling.
    for _ in range(n_steps):
        # Draw a completely new point.
        x = np.random.uniform(lower, upper)

        # Evaluate the function.
        fx =  

        # Update the best point if needed.
        if fx < best_fx:
            best_x =  
            best_fx =  

        # Save the history.
        path.append(x.copy())
        best_values.append(best_fx)

    # Return the results.
    return best_x, best_fx, path, best_values

## Example: Random Sampling on the bowl function

In [ ]:
# Run Random Sampling on the bowl function.
best_x_rs, best_fx_rs, path_rs, best_values_rs = random_search(
    bowl, lower_bowl, upper_bowl, n_steps=180
)

# Print the numerical result.
print_result("Random Sampling", best_x_rs, best_fx_rs)



# Plot the best value curve and the search path side by side.
plot_progress_and_path(bowl, lower_bowl, upper_bowl, path_rs, best_values_rs, "Random Sampling on the bowl function", best_x_rs, path_color="red")


# Greedy Local Search (GLS)

**Main idea**

Greedy Local Search tries a nearby candidate and only accepts it if it improves the objective.

At iteration $k$:
- start from the current point $x_k$
- generate a nearby candidate
- accept it only when
\begin{equation}
f(x_{\mathrm{new}}) < f(x_k)
\end{equation}

This gives a very simple local improvement strategy.

In [ ]:
# Implement Greedy Local Search with a simple local proposal.
def greedy_local_search(f, lower, upper, x0, step_size=0.4, n_steps=200):
    # Start from the initial point.
    x = np.array(x0, dtype=float)

    # Evaluate the function at the initial point.
    fx = f(x)

    # Store the current best point and value.
    best_x = x.copy()
    best_fx = fx

    # Store the visited path and the best values.
    path = [x.copy()]
    best_values = [best_fx]

    # Repeat the local search.
    for _ in range(n_steps):
        # Generate a local perturbation.
        step = step_size * np.random.randn(len(x))

        # Build the candidate point.
        candidate =  

        # Keep the candidate inside the box use clip_to_bounds function
        candidate =  

        # Evaluate the candidate.
        candidate_fx =  

        # Accept the candidate only if it improves the value.
        if candidate_fx < fx:
            x =  
            fx =  

        # Update the best point if needed.
        if fx < best_fx:
            best_x =  
            best_fx =  

        # Save the history.
        path.append(x.copy())
        best_values.append(best_fx)

    # Return the results.
    return best_x, best_fx, path, best_values

## Example: Greedy Local Search on the bowl function

In [ ]:
# Run Greedy Local Search on the bowl function.
best_x_gls, best_fx_gls, path_gls, best_values_gls = greedy_local_search(
    bowl, lower_bowl, upper_bowl, x0_bowl, step_size=0.45, n_steps=120
)

# Print the numerical result.
print_result("Greedy Local Search", best_x_gls, best_fx_gls)



# Plot the best value curve and the search path side by side.
plot_progress_and_path(bowl, lower_bowl, upper_bowl, path_gls, best_values_gls, "Greedy Local Search on the bowl function", best_x_gls, path_color="red")


# Stochastic Local Search (SLS)

**Main idea**

Stochastic Local Search is still local, but it is less rigid than Greedy Local Search.

At iteration $k$:
- generate a nearby candidate
- accept it if it improves the objective
- sometimes accept a worse point with a small fixed probability

This adds exploration to a local method.

In [ ]:
# Implement Stochastic Local Search with a simple acceptance rule.
def stochastic_local_search(f, lower, upper, x0, step_size=0.4, n_steps=200, worse_move_prob=0.15):
    # Start from the initial point.
    x = np.array(x0, dtype=float)

    # Evaluate the function at the initial point.
    fx = f(x)

    # Store the current best point and value.
    best_x = x.copy()
    best_fx = fx

    # Store the visited path and the best values.
    path = [x.copy()]
    best_values = [best_fx]

    # Repeat the local stochastic search.
    for _ in range(n_steps):
        # Generate a local perturbation.
        step = step_size * np.random.randn(len(x))

        # Build the candidate point.
        candidate =  

        # Keep the candidate inside the box use the clip_to_bounds function
        candidate =  

        # Evaluate the candidate.
        candidate_fx =  

        # Decide whether to accept the move.
        if candidate_fx < fx:
            x =  
            fx =  
        else:
            if np.random.rand() < worse_move_prob:
                x =  
                fx =  

        # Update the best point if needed.
        if fx < best_fx:
            best_x =  
            best_fx =  

        # Save the history.
        path.append(x.copy())
        best_values.append(best_fx)

    # Return the results.
    return best_x, best_fx, path, best_values

## Example: Stochastic Local Search on the bowl function

In [ ]:
# Run Stochastic Local Search on the bowl function.
best_x_sls, best_fx_sls, path_sls, best_values_sls = stochastic_local_search(
    bowl, lower_bowl, upper_bowl, x0_bowl, step_size=0.45, n_steps=120, worse_move_prob=0.12
)

# Print the numerical result.
print_result("Stochastic Local Search", best_x_sls, best_fx_sls)



# Plot the best value curve and the search path side by side.
plot_progress_and_path(bowl, lower_bowl, upper_bowl, path_sls, best_values_sls, "Stochastic Local Search on the bowl function", best_x_sls, path_color="red")


# Simulated Annealing (SA)

**Main idea**

Simulated Annealing also accepts worse moves, but not with a fixed probability.

If a candidate is worse, it is accepted with probability
\begin{equation}
P = \exp\left(-\frac{\Delta f}{T}\right),
\end{equation}
where $\Delta f$ is the increase in the objective value and $T$ is the temperature.

At high temperature the method explores more.
At low temperature the method becomes more conservative.

In [ ]:
# Implement Simulated Annealing with a simple geometric cooling rule.
def simulated_annealing(f, lower, upper, x0, step_size=0.4, n_steps=200, temperature=2.0, cooling=0.98):
    # Start from the initial point.
    x = np.array(x0, dtype=float)

    # Evaluate the function at the initial point.
    fx = f(x)

    # Store the current best point and value.
    best_x = x.copy()
    best_fx = fx

    # Store the visited path and the best values.
    path = [x.copy()]
    best_values = [best_fx]

    # Start from the initial temperature.
    T = temperature

    # Repeat the annealing steps.
    for _ in range(n_steps):
        # Generate a local perturbation.
        step = step_size * np.random.randn(len(x))

        # Build the candidate point.
        candidate = x + step

        # Keep the candidate inside the box use clip_to_bounds function
        candidate =  

        # Evaluate the candidate.
        candidate_fx =  

        # Compute the change in the objective value.
        delta =  

        # Accept all improving moves.
        if delta < 0:
            x =  
            fx =  
        else:
            # Sometimes accept a worse move with probabily delta f / T
            if np.random.rand() < .... :
                x =  
                fx =  

        # Update the best point if needed.
        if fx < best_fx:
            best_x =  
            best_fx =  

        # Save the history.
        path.append(x.copy())
        best_values.append(best_fx)

        # Decrease the temperature.
        T = cooling * T

    # Return the results.
    return best_x, best_fx, path, best_values

## Example: Simulated Annealing on the bowl function

**Choice of parameters**

The following values are used:

- $\text{step\_size} = 0.5$

  The bowl function is smooth and convex, so a slightly larger step size allows faster movement toward the minimum without the risk of jumping between multiple local minima.

- $\text{n\_steps} = 140$

  Fewer iterations are sufficient because the function is simple and does not require extensive exploration.

- $\text{temperature} = 2.5$

  A moderate initial temperature allows some flexibility in the early iterations,  
  but excessive exploration is not necessary since the function has a single global minimum.

- $\text{cooling} = 0.97$

  The temperature decreases relatively quickly:
  \begin{equation}
  T_{k+1} = 0.97 \cdot T_k
  \end{equation}
  This encourages a faster transition from exploration to exploitation.

**Interpretation**

- Early iterations: limited exploration, just enough to avoid poor initial moves.
- Later iterations: rapid convergence toward the global minimum.
- The setup is adapted to a smooth, well-behaved function where exploration is less critical.

In [ ]:
# Run Simulated Annealing on the bowl function.
best_x_sa, best_fx_sa, path_sa, best_values_sa = simulated_annealing(
    bowl, lower_bowl, upper_bowl, x0_bowl, step_size=0.5, n_steps=140, temperature=2.5, cooling=0.97
)

# Print the numerical result.
print_result("Simulated Annealing", best_x_sa, best_fx_sa)



# Plot the best value curve and the search path side by side.
plot_progress_and_path(bowl, lower_bowl, upper_bowl, path_sa, best_values_sa, "Simulated Annealing on the bowl function", best_x_sa, path_color="red")


# Particle Swarm Optimization (PSO)

**Main idea**

Particle Swarm Optimization keeps several points at the same time.

Each particle has:
- a position
- a velocity
- its own best point found so far

The swarm also stores the best point found by all particles.

The method updates the particles by mixing:
- inertia
- attraction to the particle's own best point
- attraction to the swarm's best point

In [ ]:
# Implement a very simple Particle Swarm Optimization method.
def particle_swarm_optimization(f, lower, upper, x0, n_particles=15, n_steps=80, swarm_spread=0.8, w=0.7, c1=1.5, c2=1.5):
    # Count the dimension of the problem.
    dim = len(lower)

    # Build the initial swarm around the given starting point.
    positions = x0 + swarm_spread * np.random.randn(n_particles, dim)
    positions = np.array([clip_to_bounds(p, lower, upper) for p in positions])

    # Start with small random velocities.
    velocities = 0.2 * np.random.randn(n_particles, dim)

    # Initialize each personal best with the starting position.
    personal_best_positions = positions.copy()
    personal_best_values = np.array([f(x) for x in positions])

    # Find the global best among all particles.
    best_index = np.argmin(personal_best_values)
    global_best_position = personal_best_positions[best_index].copy()
    global_best_value = personal_best_values[best_index]

    # Store the global best path and the best values.
    path = [x0.copy(), global_best_position.copy()]
    best_values = [f(x0), global_best_value]

    # Repeat the swarm updates.
    for _ in range(n_steps):
        # Update each particle.
        for i in range(n_particles):
            # Draw the random coefficients.
            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            # Update the velocity.
            velocities[i] = (
                w * velocities[i]
                + c1 * r1 * (personal_best_positions[i] - positions[i])
                + c2 * r2 * (global_best_position - positions[i])
            )

            # Update the position.
            positions[i] = positions[i] + velocities[i]
            positions[i] = clip_to_bounds(positions[i], lower, upper)

            # Evaluate the new position.
            value = f(positions[i])

            # Update the personal best if needed.
            if value < personal_best_values[i]:
                personal_best_positions[i] = positions[i].copy()
                personal_best_values[i] = value

        # Update the global best after all particles moved.
        best_index = np.argmin(personal_best_values)
        if personal_best_values[best_index] < global_best_value:
            global_best_position = personal_best_positions[best_index].copy()
            global_best_value = personal_best_values[best_index]

        # Save the history.
        path.append(global_best_position.copy())
        best_values.append(global_best_value)

    # Return the results.
    return global_best_position, global_best_value, path, best_values


## Example: Particle Swarm Optimization on the bowl function

**Choice of parameters**

The following values are used:

- $\text{n\_particles} = 15$

  A moderate number of particles provides a balance between exploration of the search space and computational cost.  
  Too few particles may miss important regions, while too many increase computation without significant benefit.

- $\text{n\_steps} = 80$

  The swarm is evolved for a sufficient number of iterations to allow convergence,  
  while keeping the runtime reasonable.

- $\text{swarm\_spread} = 0.8$

  This controls how far the initial particles are distributed around $x_0$.  
  A relatively large spread ensures that the swarm explores a broader region at the beginning.

- $w = 0.7$ (inertia weight)

  The inertia term controls how much of the previous velocity is retained.   A value less than $1$ helps stabilize the trajectories and prevents oscillations.

- $c_1 = 1.5$ (cognitive coefficient)

  This parameter controls how strongly each particle is attracted toward its own best position.  
  It promotes individual exploration and learning.

- $c_2 = 1.5$ (social coefficient)

  This parameter controls how strongly particles are attracted toward the global best position found by the swarm.  
  It promotes collective convergence.

**Interpretation**

- Early iterations: particles explore different regions due to initial spread and inertia.
- Mid iterations: balance between personal exploration ($c_1$) and group behavior ($c_2$).
- Later iterations: convergence toward the best region found by the swarm.

This setup provides a good compromise between exploration of the domain and convergence speed.

In [ ]:
# Run Particle Swarm Optimization on the bowl function.
best_x_pso, best_fx_pso, path_pso, best_values_pso = particle_swarm_optimization(
    bowl, lower_bowl, upper_bowl, x0_bowl, n_particles=18, n_steps=80, swarm_spread=0.9, w=0.7, c1=1.5, c2=1.5
)

# Print the numerical result.
print_result("Particle Swarm Optimization", best_x_pso, best_fx_pso)

# Plot the best value curve and the search path side by side.
plot_progress_and_path(bowl, lower_bowl, upper_bowl, path_pso, best_values_pso, "Particle Swarm Optimization on the bowl function", best_x_pso, path_color="red")


# Multimodal function

Now that the methods have been tested on a very simple function, we move to a more interesting landscape.

The function used here is the two-variable Rastrigin function:
\begin{equation}
f_{\mathrm{ras}}(x_1,x_2)
=
20 + (x_1^2 - 10\cos(2\pi x_1)) + (x_2^2 - 10\cos(2\pi x_2)).
\end{equation}

Its global minimum is known exactly:
\begin{equation}
x^\star=(0,0),
\qquad
f_{\mathrm{ras}}(x^\star)=0.
\end{equation}

This function has many local minima, so it is useful for comparing local and global search behavior from different initial points.

In [ ]:
# Define the Rastrigin function.
def rastrigin(x):
    x1, x2 = x
    return 20 + (x1**2 - 10 * np.cos(2 * np.pi * x1)) + (x2**2 - 10 * np.cos(2 * np.pi * x2))

# Define the search box and several initial points.
lower_ras = np.array([-5.12, -5.12])
upper_ras = np.array([5.12, 5.12])

initial_points = [
    np.array([4.0, 4.0]),
    np.array([2.8, -3.2]),
    np.array([-4.2, 1.5]),
]

In [ ]:
# Create a grid over the Rastrigin domain.
x1 = np.linspace(lower_ras[0], upper_ras[0], 300)
x2 = np.linspace(lower_ras[1], upper_ras[1], 300)
X1, X2 = np.meshgrid(x1, x2)

# Evaluate the Rastrigin function on the grid.
Z = 20 + (X1**2 - 10 * np.cos(2 * np.pi * X1)) + (X2**2 - 10 * np.cos(2 * np.pi * X2))

# Create figure with two subplots.
fig = plt.figure(figsize=(13, 5))

# ---- 2D contour plot ----
ax1 = fig.add_subplot(1, 2, 1)
ax1.contour(X1, X2, Z, levels=50)

# Plot initial points first.
for x0 in initial_points:
    ax1.scatter(x0[0], x0[1], s=120, color="orange", edgecolors="black")

# Plot global minimum LAST (on top).
ax1.scatter(0, 0, s=300, color="red", marker="*", zorder=100)
ax1.text(0.3, 0.3, r"$\mathbf{GLOBAL\ MIN}$", color="red")

ax1.set_title("Rastrigin Function (2D Contour)")
ax1.set_xlabel("$x_1$")
ax1.set_ylabel("$x_2$")

# ---- 3D surface plot ----
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(X1, X2, Z, cmap="viridis", alpha=0.9, linewidth=0)

# Plot initial points first.
for x0 in initial_points:
    z0 = rastrigin(x0)
    ax2.scatter(x0[0], x0[1], z0, s=120, color="orange", edgecolors="black")



ax2.set_title("Rastrigin Function (3D Surface)")
ax2.set_xlabel("$x_1$")
ax2.set_ylabel("$x_2$")
ax2.set_zlabel("$f(x)$")

plt.tight_layout()
plt.show()

## Example: multimodal function from different initial points

The global minimum of the Rastrigin function is at $x^*=(0,0)$ and its minimum value is $f(x^*)=0$.

In this section each method is run separately from several initial points. The left panel shows the best value curve, and the right panel shows the search path.


In [ ]:
# Define the initial points used in the multimodal example.
initial_points = [
    np.array([4.0, 4.0]),
    np.array([-4.5, 3.5]),
    np.array([2.5, -3.5]),
]

## Example: Random Walk on the multimodal function


In [ ]:
# Run Random Walk from each initial point.
for x0 in initial_points:
    best_x_rw_multi, best_fx_rw_multi, path_rw_multi, best_values_rw_multi = random_walk(
    rastrigin, lower_ras, upper_ras, x0, step_size=0.45, n_steps=180
    )

    # Print the numerical result for Random Walk.
    print_result("RW from initial point " + str(tuple(x0)), best_x_rw_multi, best_fx_rw_multi)

    # Plot the best value curve and the search path for Random Walk.
    plot_progress_and_path(rastrigin, lower_ras, upper_ras, path_rw_multi, best_values_rw_multi, "RW on the multimodal function from initial point " + str(tuple(x0)), best_x_rw_multi, path_color="red")


## Example: Random Sampling on the multimodal function


In [ ]:
# Run Random Sampling from each initial point.
for x0 in initial_points:
    best_x_rs_multi, best_fx_rs_multi, path_rs_multi, best_values_rs_multi = random_search(
    rastrigin, lower_ras, upper_ras, n_steps=220
    )

    # Print the numerical result for Random Sampling.
    print_result("RS from initial point " + str(tuple(x0)), best_x_rs_multi, best_fx_rs_multi)

    # Plot the best value curve and the search path for Random Sampling.
    plot_progress_and_path(rastrigin, lower_ras, upper_ras, path_rs_multi, best_values_rs_multi, "RS on the multimodal function from initial point " + str(tuple(x0)), best_x_rs_multi, path_color="red")


## Example: Greedy Local Search on the multimodal function

In [ ]:
# Run Greedy Local Search from each initial point.
for x0 in initial_points:
    best_x_gls_multi, best_fx_gls_multi, path_gls_multi, best_values_gls_multi = greedy_local_search(
    rastrigin, lower_ras, upper_ras, x0, step_size=0.35, n_steps=180
    )

    # Print the numerical result for Greedy Local Search.
    print_result("GLS from initial point " + str(tuple(x0)), best_x_gls_multi, best_fx_gls_multi)

    # Plot the best value curve and the search path for Greedy Local Search.
    plot_progress_and_path(rastrigin, lower_ras, upper_ras, path_gls_multi, best_values_gls_multi, "GLS on the multimodal function from initial point " + str(tuple(x0)), best_x_gls_multi, path_color="red")


In [ ]:
# Run Stochastic Local Search from each initial point.
for x0 in initial_points:
    best_x_sls_multi, best_fx_sls_multi, path_sls_multi, best_values_sls_multi = stochastic_local_search(
    rastrigin, lower_ras, upper_ras, x0, step_size=0.35, n_steps=180, worse_move_prob=0.15
    )

    # Print the numerical result for Stochastic Local Search.
    print_result("SLS from initial point " + str(tuple(x0)), best_x_sls_multi, best_fx_sls_multi)

    # Plot the best value curve and the search path for Stochastic Local Search.
    plot_progress_and_path(rastrigin, lower_ras, upper_ras, path_sls_multi, best_values_sls_multi, "SLS on the multimodal function from initial point " + str(tuple(x0)), best_x_sls_multi, path_color="red")


## Example: Simulated Annealing on the multimodal function

In [ ]:
# Run Simulated Annealing from each initial point.
for x0 in initial_points:
    best_x_sa_multi, best_fx_sa_multi, path_sa_multi, best_values_sa_multi = simulated_annealing(
    rastrigin, lower_ras, upper_ras, x0, step_size=0.4, n_steps=220, temperature=4.0, cooling=0.985
    )

    # Print the numerical result for Simulated Annealing.
    print_result("SA from initial point " + str(tuple(x0)), best_x_sa_multi, best_fx_sa_multi)

    # Plot the best value curve and the search path for Simulated Annealing.
    plot_progress_and_path(rastrigin, lower_ras, upper_ras, path_sa_multi, best_values_sa_multi, "SA on the multimodal function from initial point " + str(tuple(x0)), best_x_sa_multi, path_color="red")


## Example: Particle Swarm Optimization on the multimodal function


In [ ]:
# Run Particle Swarm Optimization from each initial point.
for x0 in initial_points:
    best_x_pso_multi, best_fx_pso_multi, path_pso_multi, best_values_pso_multi = particle_swarm_optimization(
    rastrigin, lower_ras, upper_ras, x0, n_particles=22, n_steps=100, swarm_spread=0.8, w=0.7, c1=1.5, c2=1.5
    )

    # Print the numerical result for Particle Swarm Optimization.
    print_result("PSO from initial point " + str(tuple(x0)), best_x_pso_multi, best_fx_pso_multi)

    # Plot the best value curve and the search path for Particle Swarm Optimization.
    plot_progress_and_path(rastrigin, lower_ras, upper_ras, path_pso_multi, best_values_pso_multi, "PSO on the multimodal function from initial point " + str(tuple(x0)), best_x_pso_multi, path_color="red")


# Varying the neighborhood size

For the methods that use local perturbations, the neighborhood size is controlled here by `step_size`.

Typical questions to explore:
- What happens if `step_size` is much smaller?
- What happens if `step_size` is much larger?
- Does a larger neighborhood help exploration on the Rastrigin function?
- Does a smaller neighborhood help refinement near a minimizer?

Methods where this is especially relevant:
- Random Walk
- Greedy Local Search
- Stochastic Local Search
- Simulated Annealing

Try several values and compare the best value curves and the search paths. In particular, compare very small neighborhoods, moderate neighborhoods, and very large neighborhoods.

In [ ]:
# Example: compare different neighborhood sizes for Greedy Local Search on Rastrigin.
step_sizes = [0.1, 0.3, 0.7, 1.2]
x0_test = np.array([4.0, 4.0])

plt.figure(figsize=(10, 5))

for step_size in step_sizes:
    best_x, best_fx, path, best_values = greedy_local_search(
        rastrigin, lower_ras, upper_ras, x0_test, step_size=step_size, n_steps=180
    )
    plt.plot(best_values, label=f"step_size = {step_size}")

plt.xlabel("iteration")
plt.ylabel("best value so far")
plt.title("Greedy Local Search on Rastrigin for different neighborhood sizes")
plt.legend()
plt.show()

# Final remarks

- On the bowl function, all methods can approach the minimum because the landscape is simple.
- On the multimodal function, the behavior changes substantially.
- The initial point matters much more for local methods.
- Exploration mechanisms become more important when many local minima are present.
- The neighborhood size strongly affects the balance between exploration and local refinement.